In [ ]:
-- =====================================================
-- LAB | SQL Subqueries - Individual Queries
-- Copy and paste each query separately into your SQL client
-- =====================================================

-- QUERY 1: Number of copies of "Hunchback Impossible"
SELECT COUNT(*) as number_of_copies
FROM inventory i
JOIN film f ON i.film_id = f.film_id
WHERE f.title = 'Hunchback Impossible';

-- =====================================================

-- QUERY 2: Films longer than average length
SELECT title, length
FROM film
WHERE length > (
    SELECT AVG(length) 
    FROM film
)
ORDER BY length DESC;

-- =====================================================

-- QUERY 3: Actors who appear in "Alone Trip"
SELECT a.first_name, a.last_name
FROM actor a
WHERE a.actor_id IN (
    SELECT fa.actor_id
    FROM film_actor fa
    JOIN film f ON fa.film_id = f.film_id
    WHERE f.title = 'Alone Trip'
);

-- =====================================================

-- BONUS 1: Family films
SELECT f.title
FROM film f
JOIN film_category fc ON f.film_id = fc.film_id
JOIN category c ON fc.category_id = c.category_id
WHERE c.name = 'Family'
ORDER BY f.title;

-- =====================================================

-- BONUS 2A: Canadian customers (using joins)
SELECT c.first_name, c.last_name, c.email
FROM customer c
JOIN address a ON c.address_id = a.address_id
JOIN city ci ON a.city_id = ci.city_id
JOIN country co ON ci.country_id = co.country_id
WHERE co.country = 'Canada';

-- =====================================================

-- BONUS 2B: Canadian customers (using subqueries)
SELECT first_name, last_name, email
FROM customer
WHERE address_id IN (
    SELECT address_id
    FROM address
    WHERE city_id IN (
        SELECT city_id
        FROM city
        WHERE country_id = (
            SELECT country_id
            FROM country
            WHERE country = 'Canada'
        )
    )
);

-- =====================================================

-- BONUS 3: Films by most prolific actor
SELECT f.title
FROM film f
JOIN film_actor fa ON f.film_id = fa.film_id
WHERE fa.actor_id = (
    SELECT actor_id
    FROM film_actor
    GROUP BY actor_id
    ORDER BY COUNT(*) DESC
    LIMIT 1
)
ORDER BY f.title;

-- =====================================================

-- BONUS 4: Films rented by most profitable customer
SELECT DISTINCT f.title
FROM film f
JOIN inventory i ON f.film_id = i.film_id
JOIN rental r ON i.inventory_id = r.inventory_id
WHERE r.customer_id = (
    SELECT customer_id
    FROM payment
    GROUP BY customer_id
    ORDER BY SUM(amount) DESC
    LIMIT 1
)
ORDER BY f.title;

-- =====================================================

-- BONUS 5: Customers spending above average
SELECT customer_id, total_amount_spent
FROM (
    SELECT customer_id, SUM(amount) as total_amount_spent
    FROM payment
    GROUP BY customer_id
) as customer_spending
WHERE total_amount_spent > (
    SELECT AVG(customer_total)
    FROM (
        SELECT customer_id, SUM(amount) as customer_total
        FROM payment
        GROUP BY customer_id
    ) as avg_calculation
)
ORDER BY total_amount_spent DESC;